# 04 — Training

1. Train a quick **baseline** (`LinearRegression`) on the preprocessed splits.
2. Run the package's `run_training` to fit a **tuned** model (Ridge, GridSearchCV) into a notebook-local artifact dir — keeps the main `./artifacts/` clean.

Production code path: `prices.train.trainer.run_training`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SCRATCH = REPO_ROOT / "notebooks" / "_artifacts"

import json
import logging

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

X_train_scaled = np.load(SCRATCH / "X_train_scaled.npy")
X_test_scaled = np.load(SCRATCH / "X_test_scaled.npy")
y_train = np.load(SCRATCH / "y_train.npy")
y_test = np.load(SCRATCH / "y_test.npy")

print(f"X_train: {X_train_scaled.shape}")
print(f"X_test:  {X_test_scaled.shape}")

X_train: (7997, 11)
X_test:  (2000, 11)


## Baseline — LinearRegression

In [2]:
baseline = LinearRegression().fit(X_train_scaled, y_train)
preds = baseline.predict(X_test_scaled)

baseline_mse = mean_squared_error(y_test, preds)
baseline_r2 = r2_score(y_test, preds)
print(f"baseline test MSE: {baseline_mse:>15,.0f}")
print(f"baseline test R^2: {baseline_r2:>15.4f}")

baseline test MSE:   4,927,999,509
baseline test R^2:          0.6699


## Tuned model — Ridge via package trainer

In [3]:
from prices.train.trainer import run_training

results = run_training(out_dir=SCRATCH, models=["ridge"])
print(json.dumps(results, indent=2, default=str))

INFO prices.data.ingestion: Dropped 3 rows (NaNs/duplicates) from /home/nick/house-prices/data/raw/dataset.csv


INFO prices.train.trainer: Training ridge


Fitting 4 folds for each of 24 candidates, totalling 96 fits


INFO prices.train.trainer: ridge — test MSE=4927995267.47, R2=0.6699


INFO prices.train.trainer: Best by CV score: ridge — running stratified OOF CV


INFO prices.train.trainer: ridge OOF — MSE=7469242071.01, RMSE=86424.78, R2=0.6584


{
  "per_model": {
    "ridge": {
      "best_params": {
        "alpha": 0.01,
        "solver": "auto"
      },
      "cv_score": -7406106801.603244,
      "test_mse": 4927995267.468864,
      "test_r2": 0.6698609281871042
    }
  },
  "oof_best": {
    "model": "ridge",
    "n_splits": 5,
    "n_bins": 10,
    "oof_mse": 7469242071.012484,
    "oof_rmse": 86424.77695089807,
    "oof_r2": 0.658419107636304
  }
}


In [4]:
for f in sorted(SCRATCH.iterdir()):
    print(f"{f.name:<25} {f.stat().st_size:>10,} bytes")

X_test_raw.parquet            80,871 bytes
X_test_scaled.npy            176,128 bytes
X_train_scaled.npy           703,864 bytes
clean.parquet                423,335 bytes
feature_names.json               116 bytes
ridge.joblib                     641 bytes
scaler.joblib                  1,183 bytes
summary.json                     417 bytes
y_test.npy                    16,128 bytes
y_train.npy                   64,104 bytes
